# VR Firing Rate Profile — PCA / UMAP / t-SNE

## Purpose
Unbiased characterisation of spatial tuning across the MEC medial–lateral axis
using the shape of each cell's VR average firing rate profile.

Rather than relying on open-field cell-type labels, we:

1. Compute the **average firing rate profile** (all trials) for every MEC principal cell
2. **Z-score normalise** each profile (per cell) so shape — not rate — drives variation
3. Run **PCA** to find the dominant modes of profile shape
4. Run **UMAP** and **t-SNE** for non-linear low-dimensional embeddings
5. Ask whether any embedding axis correlates with **ML position**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.ndimage import gaussian_filter1d
from scipy.stats import spearmanr, zscore as scipy_zscore
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '/Users/harryclark/Documents/spatial-manifolds/src')
import pynapple as nap
from spatial_manifolds.detect_grids import curate_clusters

try:
    import umap as umap_lib
    HAS_UMAP = True
    print('umap-learn found')
except ImportError:
    HAS_UMAP = False
    print('umap-learn not installed — UMAP cell will be skipped')

In [ ]:
source_path         = '/Users/harryclark/Downloads/COHORT12/'
CLASSIFICATIONS_CSV = '/Users/harryclark/Documents/spatial-manifolds/data/cell_classifications.csv'
PROFILE_CACHE       = '/Users/harryclark/Documents/spatial-manifolds/data/eddie/vr_avg_profiles.parquet'

N_BINS    = 40
SIGMA     = 1.5
TRACK_LEN = 200.0
ML_SPLIT  = 3400   # µm — medial boundary

mouse_days = {
    20: [14,15,16,17,18,19,20,21,22,23,24,25,26],
    21: [15,16,17,18,19,20,21,22,23,24,25,26],
    22: [33,34,35,36,37,38,39,40,41],
    25: [16,17,18,19,20,21,22,23,24,25],
    26: [11,12,13,14,15,16,17,18,19],
    27: [16,17,18,19,20,21,22,23,24,26],
    28: [16,17,18,19,20,21,22,23,25],
    29: [16,17,18,19,20,21,22,23,25],
}

BIN_CENTRES = np.linspace(TRACK_LEN / N_BINS / 2, TRACK_LEN - TRACK_LEN / N_BINS / 2, N_BINS)
BIN_COLS    = [f'bin_{i}' for i in range(N_BINS)]

TYPE_COLOURS = {'GC': '#E53935', 'NG': '#1565C0', 'Other': '#78909C'}
TYPE_ORDER   = ['GC', 'NG', 'Other']

In [ ]:
def load_vr_session(mouse, day, source_path):
    sf = f'{source_path}M{mouse}/D{day:02}/VR/'
    beh      = nap.load_file(sf + f'sub-{mouse}_day-{day:02}_ses-VR_beh.nwb')
    clusters = curate_clusters(
        nap.load_file(sf + f'sub-{mouse}_day-{day:02}_ses-VR_srt-kilosort4_clusters.npz')
    )
    return beh, clusters


def compute_avg_profile(cluster_ts, position, ep,
                        n_bins=N_BINS, track_len=TRACK_LEN, sigma=SIGMA):
    """Occupancy-corrected average firing rate profile across all trials in ep."""
    tc = nap.compute_1d_tuning_curves(
        nap.TsGroup([cluster_ts]), position,
        nb_bins=n_bins, minmax=[0.0, track_len], ep=ep,
    )[0]
    return gaussian_filter1d(np.nan_to_num(np.array(tc, dtype=float)), sigma=sigma)

In [ ]:
# ── Build profile matrix — skip if cache exists ───────────────────────────────
cache_path = Path(PROFILE_CACHE)
cache_path.parent.mkdir(parents=True, exist_ok=True)

if cache_path.exists():
    print(f'Loading cached profiles from {PROFILE_CACHE}')
    df_profiles = pd.read_parquet(PROFILE_CACHE)
else:
    rows = []
    for mouse, days in mouse_days.items():
        for day in days:
            try:
                beh, clusters = load_vr_session(mouse, day, source_path)
            except Exception as e:
                print(f'  M{mouse}D{day}: {e}')
                continue
            position = beh['P']
            ep_all   = beh['trials']   # all trial types
            for cid in clusters.index:
                prof = compute_avg_profile(clusters[cid], position, ep_all)
                row  = dict(mouse=int(mouse), day=int(day), cluster_id=int(cid),
                            brain_region=str(clusters.brain_region[cid]),
                            mean_rate=float(prof.mean()))
                row.update({f'bin_{i}': float(v) for i, v in enumerate(prof)})
                rows.append(row)
            print(f'  M{mouse}D{day}: {len(clusters)} cells')

    df_profiles = pd.DataFrame(rows)
    df_profiles.to_parquet(PROFILE_CACHE, index=False)
    print(f'\nSaved {len(df_profiles)} profiles → {PROFILE_CACHE}')

print(f'Total profiles: {len(df_profiles)}')

In [ ]:
df_class = pd.read_csv(CLASSIFICATIONS_CSV)[
    ['mouse','day','cluster_id','cell_type','firing_rate_VR','SC_x']
].copy()
df_class['mouse'] = df_class['mouse'].astype(int)
df_class['day']   = df_class['day'].astype(int)
df_class['ml']    = df_class['SC_x'].abs()

df = df_profiles.merge(df_class, on=['mouse','day','cluster_id'], how='inner')

# MEC principal cells only
df = df[
    df['brain_region'].str.startswith('ENTm', na=False) &
    (df['firing_rate_VR'] < 10)
].copy()
df = df.reset_index(drop=True)

print(f'MEC principal cells: {len(df)}')
print(df['cell_type'].value_counts().to_string())
print(f'ML range: {df["ml"].min():.0f} – {df["ml"].max():.0f} µm')

In [ ]:
profiles_raw = df[BIN_COLS].values.astype(float)

# Per-cell z-score: subtract mean and divide by std of that cell's profile
profile_means = profiles_raw.mean(axis=1, keepdims=True)
profile_stds  = profiles_raw.std(axis=1, keepdims=True)

# Exclude cells with near-flat profiles (std < 0.01 Hz — no meaningful shape)
MIN_STD = 0.01
valid   = (profile_stds.ravel() >= MIN_STD)
print(f'Cells with flat profiles excluded: {(~valid).sum()} / {len(df)}')

df_z       = df[valid].copy().reset_index(drop=True)
X_raw      = profiles_raw[valid]
X_z        = (X_raw - profile_means[valid]) / profile_stds[valid]

print(f'Cells entering embedding: {len(df_z)}  '
      f'({df_z["cell_type"].value_counts().to_dict()})')

In [ ]:
N_COMPONENTS = 20

pca  = PCA(n_components=N_COMPONENTS, random_state=42)
X_pc = pca.fit_transform(X_z)

# Store PC scores in df_z
for i in range(N_COMPONENTS):
    df_z[f'PC{i+1}'] = X_pc[:, i]

# ── Scree plot ────────────────────────────────────────────────────────────────
var_exp     = pca.explained_variance_ratio_
var_exp_cum = np.cumsum(var_exp)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5),
                         gridspec_kw=dict(wspace=0.35))

ax = axes[0]
ax.bar(range(1, N_COMPONENTS+1), var_exp * 100, color='#455A64', alpha=0.85, lw=0)
ax.set_xlabel('Principal component', fontsize=9)
ax.set_ylabel('Variance explained (%)', fontsize=9)
ax.set_title('Scree plot', fontsize=9, fontweight='bold')
ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=7)

ax = axes[1]
ax.plot(range(1, N_COMPONENTS+1), var_exp_cum * 100, 'o-',
        color='#1565C0', ms=5, lw=1.6)
ax.axhline(80, color='#EF5350', lw=0.9, ls='--', label='80%')
ax.axhline(90, color='#FB8C00', lw=0.9, ls='--', label='90%')
ax.set_xlabel('N components', fontsize=9)
ax.set_ylabel('Cumulative variance (%)', fontsize=9)
ax.set_title('Cumulative variance explained', fontsize=9, fontweight='bold')
ax.legend(fontsize=8, frameon=False)
ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=7)

fig.suptitle('PCA on z-scored VR firing rate profiles', fontsize=10, fontweight='bold')
plt.savefig('profile_pca_scree.pdf', bbox_inches='tight', dpi=150)
plt.show()
print(f'PC1={var_exp[0]*100:.1f}%  PC2={var_exp[1]*100:.1f}%  '
      f'PC3={var_exp[2]*100:.1f}%  (cumulative PC1-3={var_exp_cum[2]*100:.1f}%)')

In [ ]:
# ── PC loadings: what spatial patterns do the PCs capture? ───────────────────
N_SHOW = 6
fig, axes = plt.subplots(2, N_SHOW // 2, figsize=(14, 5),
                         gridspec_kw=dict(hspace=0.55, wspace=0.30))
for i, ax in enumerate(axes.flat):
    loading = pca.components_[i]
    ax.fill_between(BIN_CENTRES, loading, alpha=0.45, color='#37474F', lw=0)
    ax.plot(BIN_CENTRES, loading, color='#1A237E', lw=1.0)
    ax.axhline(0, color='#90A4AE', lw=0.6, ls=':')
    ax.set_title(f'PC{i+1}  ({var_exp[i]*100:.1f}%)', fontsize=9, fontweight='bold')
    ax.set_xlabel('Position (cm)', fontsize=7.5)
    ax.set_ylabel('Loading', fontsize=7.5)
    ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=6.5)

fig.suptitle('PCA loadings — spatial patterns captured per component',
             fontsize=10, fontweight='bold')
plt.savefig('profile_pca_loadings.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
def scatter_embedding(ax, x, y, c_vals, c_map, label, title, vmin=None, vmax=None, s=12):
    sc = ax.scatter(x, y, c=c_vals, cmap=c_map, s=s, alpha=0.65, lw=0,
                    vmin=vmin, vmax=vmax)
    ax.set_xlabel(label[0], fontsize=8.5)
    ax.set_ylabel(label[1], fontsize=8.5)
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=7)
    return sc


fig, axes = plt.subplots(2, 3, figsize=(15, 9),
                         gridspec_kw=dict(hspace=0.48, wspace=0.35))

# ── Row 0: PC1 vs PC2 ────────────────────────────────────────────────────────
# Colour by ML position
sc = scatter_embedding(axes[0,0], df_z['PC1'], df_z['PC2'],
                       df_z['ml'], 'viridis_r', ['PC1', 'PC2'],
                       'PC1 vs PC2 — coloured by ML position (µm)')
plt.colorbar(sc, ax=axes[0,0], label='ML (µm)', shrink=0.85)

# Colour by cell type
for ct in TYPE_ORDER:
    mask = df_z['cell_type'] == ct
    axes[0,1].scatter(df_z.loc[mask,'PC1'], df_z.loc[mask,'PC2'],
                      s=12, color=TYPE_COLOURS[ct], alpha=0.65, lw=0, label=ct)
axes[0,1].set_xlabel('PC1', fontsize=8.5); axes[0,1].set_ylabel('PC2', fontsize=8.5)
axes[0,1].set_title('PC1 vs PC2 — coloured by cell type', fontsize=9, fontweight='bold')
axes[0,1].legend(fontsize=8, frameon=False, markerscale=1.5)
axes[0,1].spines[['top','right']].set_visible(False); axes[0,1].tick_params(labelsize=7)

# Colour by mouse
mice = df_z['mouse'].unique()
cmap_m = plt.cm.get_cmap('tab10', len(mice))
for mi, m in enumerate(sorted(mice)):
    mask = df_z['mouse'] == m
    axes[0,2].scatter(df_z.loc[mask,'PC1'], df_z.loc[mask,'PC2'],
                      s=12, color=cmap_m(mi), alpha=0.65, lw=0, label=f'M{m}')
axes[0,2].set_xlabel('PC1', fontsize=8.5); axes[0,2].set_ylabel('PC2', fontsize=8.5)
axes[0,2].set_title('PC1 vs PC2 — coloured by mouse', fontsize=9, fontweight='bold')
axes[0,2].legend(fontsize=7, frameon=False, markerscale=1.5, ncol=2)
axes[0,2].spines[['top','right']].set_visible(False); axes[0,2].tick_params(labelsize=7)

# ── Row 1: PC2 vs PC3, PC1 vs PC3, PC1 vs PC2 NG only ───────────────────────
sc2 = scatter_embedding(axes[1,0], df_z['PC2'], df_z['PC3'],
                        df_z['ml'], 'viridis_r', ['PC2', 'PC3'],
                        'PC2 vs PC3 — ML position')
plt.colorbar(sc2, ax=axes[1,0], label='ML (µm)', shrink=0.85)

sc3 = scatter_embedding(axes[1,1], df_z['PC1'], df_z['PC3'],
                        df_z['ml'], 'viridis_r', ['PC1', 'PC3'],
                        'PC1 vs PC3 — ML position')
plt.colorbar(sc3, ax=axes[1,1], label='ML (µm)', shrink=0.85)

# NG only
ng_mask = df_z['cell_type'] == 'NG'
sc4 = axes[1,2].scatter(df_z.loc[ng_mask,'PC1'], df_z.loc[ng_mask,'PC2'],
                         c=df_z.loc[ng_mask,'ml'], cmap='viridis_r',
                         s=14, alpha=0.75, lw=0)
plt.colorbar(sc4, ax=axes[1,2], label='ML (µm)', shrink=0.85)
axes[1,2].set_xlabel('PC1', fontsize=8.5); axes[1,2].set_ylabel('PC2', fontsize=8.5)
axes[1,2].set_title('PC1 vs PC2 — NG cells only, ML position',
                    fontsize=9, fontweight='bold')
axes[1,2].spines[['top','right']].set_visible(False); axes[1,2].tick_params(labelsize=7)

fig.suptitle('PCA embeddings of z-scored VR firing rate profiles',
             fontsize=11, fontweight='bold')
plt.savefig('profile_pca_scatter.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── Spearman correlation of PC scores with ML position ───────────────────────
N_PC_TEST = 10
rhos, pvals = [], []
for i in range(1, N_PC_TEST + 1):
    r, p = spearmanr(df_z['ml'], df_z[f'PC{i}'])
    rhos.append(r); pvals.append(p)

fig, axes = plt.subplots(1, 2, figsize=(12, 4),
                         gridspec_kw=dict(wspace=0.38))

# Bar chart of Spearman rho per PC
ax = axes[0]
cols = ['#E53935' if p < 0.05 else '#90A4AE' for p in pvals]
ax.bar(range(1, N_PC_TEST+1), rhos, color=cols, alpha=0.85, lw=0)
ax.axhline(0, color='k', lw=0.6)
ax.set_xlabel('Principal component', fontsize=9)
ax.set_ylabel('Spearman ρ  (PC score vs ML)', fontsize=9)
ax.set_title('PC–ML correlation  (red = p<0.05)', fontsize=9, fontweight='bold')
ax.set_xticks(range(1, N_PC_TEST+1))
ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=7)
for xi, (rho, p) in enumerate(zip(rhos, pvals), 1):
    ax.text(xi, rho + 0.005 * np.sign(rho), f'p={p:.3f}',
            ha='center', va='bottom' if rho >= 0 else 'top', fontsize=5.5, rotation=90)

# Scatter of strongest ML-correlated PC vs ML
best_i  = int(np.argmax(np.abs(rhos))) + 1
ax = axes[1]
sc = ax.scatter(df_z['ml'], df_z[f'PC{best_i}'],
                c=df_z['ml'], cmap='viridis_r', s=12, alpha=0.55, lw=0)
# Per-cell-type medians in ML bins
N_BINS_ML = 8
ml_edges   = np.linspace(df_z['ml'].quantile(0.02), df_z['ml'].quantile(0.98), N_BINS_ML+1)
ml_centres = 0.5*(ml_edges[:-1]+ml_edges[1:])
for ct in ['GC','NG']:
    sub = df_z[df_z['cell_type']==ct]
    meds = [sub.loc[(sub['ml']>=lo)&(sub['ml']<hi), f'PC{best_i}'].median()
            for lo, hi in zip(ml_edges[:-1], ml_edges[1:])]
    ax.plot(ml_centres, meds, 'o-', color=TYPE_COLOURS[ct], lw=1.8, ms=5, label=ct)
ax.set_xlabel('ML position (µm)', fontsize=9)
ax.set_ylabel(f'PC{best_i} score', fontsize=9)
ax.set_title(f'PC{best_i} vs ML  (strongest corr: ρ={rhos[best_i-1]:.2f})',
             fontsize=9, fontweight='bold')
ax.legend(fontsize=8, frameon=False)
ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=7)

fig.suptitle('ML gradient in PCA space', fontsize=10, fontweight='bold')
plt.savefig('profile_pca_ml_gradient.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
if not HAS_UMAP:
    print('Skipping UMAP — install with: pip install umap-learn')
else:
    reducer  = umap_lib.UMAP(n_components=2, n_neighbors=15,
                              min_dist=0.1, random_state=42, verbose=False)
    X_umap   = reducer.fit_transform(X_z)
    df_z['UMAP1'] = X_umap[:, 0]
    df_z['UMAP2'] = X_umap[:, 1]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5),
                             gridspec_kw=dict(wspace=0.35))

    # ML position
    sc = axes[0].scatter(df_z['UMAP1'], df_z['UMAP2'],
                         c=df_z['ml'], cmap='viridis_r', s=10, alpha=0.65, lw=0)
    plt.colorbar(sc, ax=axes[0], label='ML (µm)', shrink=0.85)
    axes[0].set_title('UMAP — ML position', fontsize=9, fontweight='bold')

    # Cell type
    for ct in TYPE_ORDER:
        mask = df_z['cell_type'] == ct
        axes[1].scatter(df_z.loc[mask,'UMAP1'], df_z.loc[mask,'UMAP2'],
                        s=10, color=TYPE_COLOURS[ct], alpha=0.65, lw=0, label=ct)
    axes[1].set_title('UMAP — cell type', fontsize=9, fontweight='bold')
    axes[1].legend(fontsize=8, frameon=False)

    # NG only — ML
    ng = df_z[df_z['cell_type']=='NG']
    sc2 = axes[2].scatter(ng['UMAP1'], ng['UMAP2'],
                          c=ng['ml'], cmap='viridis_r', s=12, alpha=0.75, lw=0)
    plt.colorbar(sc2, ax=axes[2], label='ML (µm)', shrink=0.85)
    axes[2].set_title('UMAP — NG cells only, ML position', fontsize=9, fontweight='bold')

    for ax in axes:
        ax.set_xlabel('UMAP 1', fontsize=8.5); ax.set_ylabel('UMAP 2', fontsize=8.5)
        ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=7)

    # Spearman of UMAP axes vs ML
    r1, p1 = spearmanr(df_z['ml'], df_z['UMAP1'])
    r2, p2 = spearmanr(df_z['ml'], df_z['UMAP2'])
    fig.suptitle(f'UMAP (n_neighbors=15) — UMAP1 vs ML: ρ={r1:.2f} p={p1:.3f} | '
                 f'UMAP2 vs ML: ρ={r2:.2f} p={p2:.3f}',
                 fontsize=9, fontweight='bold')
    plt.savefig('profile_umap.pdf', bbox_inches='tight', dpi=150)
    plt.show()

In [ ]:
# Run t-SNE on first 10 PCs (faster + avoids curse of dimensionality)
X_tsne_in = X_pc[:, :10]

tsne    = TSNE(n_components=2, perplexity=30, n_iter=1000,
               random_state=42, init='pca', learning_rate='auto')
X_tsne  = tsne.fit_transform(X_tsne_in)
df_z['tSNE1'] = X_tsne[:, 0]
df_z['tSNE2'] = X_tsne[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5),
                         gridspec_kw=dict(wspace=0.35))

# ML position
sc = axes[0].scatter(df_z['tSNE1'], df_z['tSNE2'],
                     c=df_z['ml'], cmap='viridis_r', s=10, alpha=0.65, lw=0)
plt.colorbar(sc, ax=axes[0], label='ML (µm)', shrink=0.85)
axes[0].set_title('t-SNE — ML position', fontsize=9, fontweight='bold')

# Cell type
for ct in TYPE_ORDER:
    mask = df_z['cell_type'] == ct
    axes[1].scatter(df_z.loc[mask,'tSNE1'], df_z.loc[mask,'tSNE2'],
                    s=10, color=TYPE_COLOURS[ct], alpha=0.65, lw=0, label=ct)
axes[1].set_title('t-SNE — cell type', fontsize=9, fontweight='bold')
axes[1].legend(fontsize=8, frameon=False)

# NG only — ML
ng = df_z[df_z['cell_type']=='NG']
sc2 = axes[2].scatter(ng['tSNE1'], ng['tSNE2'],
                      c=ng['ml'], cmap='viridis_r', s=12, alpha=0.75, lw=0)
plt.colorbar(sc2, ax=axes[2], label='ML (µm)', shrink=0.85)
axes[2].set_title('t-SNE — NG cells only, ML position', fontsize=9, fontweight='bold')

for ax in axes:
    ax.set_xlabel('t-SNE 1', fontsize=8.5); ax.set_ylabel('t-SNE 2', fontsize=8.5)
    ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=7)

r1, p1 = spearmanr(df_z['ml'], df_z['tSNE1'])
r2, p2 = spearmanr(df_z['ml'], df_z['tSNE2'])
fig.suptitle(f't-SNE (perplexity=30, init=pca) — t-SNE1 vs ML: ρ={r1:.2f} p={p1:.3f} | '
             f't-SNE2 vs ML: ρ={r2:.2f} p={p2:.3f}',
             fontsize=9, fontweight='bold')
plt.savefig('profile_tsne.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── What do cells in different PC1 / PC2 quadrants look like? ────────────────
# Bin PC1 into quartiles and show mean reconstructed (raw z-scored) profile per bin
N_BINS_PC = 4
pc1_edges  = np.percentile(df_z['PC1'], np.linspace(0, 100, N_BINS_PC + 1))
fig, axes  = plt.subplots(2, N_BINS_PC, figsize=(13, 5),
                          gridspec_kw=dict(hspace=0.55, wspace=0.28))

for bi in range(N_BINS_PC):
    lo, hi  = pc1_edges[bi], pc1_edges[bi+1]
    mask    = (df_z['PC1'] >= lo) & (df_z['PC1'] < hi)
    sub_X   = X_z[mask.values]
    mean_pr = sub_X.mean(axis=0)
    sem_pr  = sub_X.std(axis=0) / np.sqrt(sub_X.shape[0])

    for row, (ct, c_filt) in enumerate([('GC', 'GC'), ('NG', 'NG')]):
        ax   = axes[row, bi]
        ct_m = (df_z['cell_type'] == ct) & mask
        if ct_m.sum() > 0:
            ct_pr  = X_z[ct_m.values].mean(axis=0)
            ct_sem = X_z[ct_m.values].std(axis=0) / np.sqrt(ct_m.sum())
            ax.fill_between(BIN_CENTRES, ct_pr - ct_sem, ct_pr + ct_sem,
                            alpha=0.25, color=TYPE_COLOURS[ct], lw=0)
            ax.plot(BIN_CENTRES, ct_pr, color=TYPE_COLOURS[ct], lw=1.2)
        ax.axhline(0, color='#90A4AE', lw=0.5, ls=':')
        ax.set_title(f'{ct}  PC1 Q{bi+1}
(n={ct_m.sum()})',
                     fontsize=7.5, fontweight='bold', color=TYPE_COLOURS[ct])
        ax.set_xlabel('Position (cm)', fontsize=7)
        if bi == 0: ax.set_ylabel('z-scored rate', fontsize=7)
        ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=6)

fig.suptitle('Mean z-scored profile per PC1 quartile (GC top, NG bottom)',
             fontsize=10, fontweight='bold')
plt.savefig('profile_pca_quartile_profiles.pdf', bbox_inches='tight', dpi=150)
plt.show()

---
## What is actually changing across the ML axis?

### 1 — Mean z-scored profile per ML bin

Bin cells by ML position and directly show how the average firing rate profile shape
changes from lateral → medial MEC, separately for GC and NG.

In [ ]:
N_ML_BINS  = 6
ml_q_edges = np.percentile(df_z['ml'], np.linspace(0, 100, N_ML_BINS + 1))
ml_labels  = [f'{ml_q_edges[i]:.0f}–{ml_q_edges[i+1]:.0f} µm'
               for i in range(N_ML_BINS)]
ml_cmap    = plt.cm.get_cmap('viridis_r', N_ML_BINS)

GROUPS = [('All MEC principal', df_z),
          ('GC',                df_z[df_z['cell_type'] == 'GC']),
          ('NG',                df_z[df_z['cell_type'] == 'NG'])]

fig, axes = plt.subplots(2, 3, figsize=(15, 8),
                         gridspec_kw=dict(hspace=0.50, wspace=0.30))

for col, (grp_label, grp) in enumerate(GROUPS):
    # ── Row 0: lines per ML bin ───────────────────────────────────────────────
    ax = axes[0, col]
    for bi in range(N_ML_BINS):
        lo, hi = ml_q_edges[bi], ml_q_edges[bi + 1]
        mask   = (grp['ml'] >= lo) & (grp['ml'] < hi)
        sub_X  = X_z[grp.index[mask.values], :]  # index into full X_z array
        if mask.sum() < 3:
            continue
        mean_pr = sub_X.mean(axis=0)
        sem_pr  = sub_X.std(axis=0) / np.sqrt(mask.sum())
        c       = ml_cmap(bi)
        ax.fill_between(BIN_CENTRES, mean_pr - sem_pr, mean_pr + sem_pr,
                        alpha=0.18, color=c, lw=0)
        ax.plot(BIN_CENTRES, mean_pr, color=c, lw=1.5,
                label=f'{ml_labels[bi]}  (n={mask.sum()})')
    ax.axhline(0, color='#90A4AE', lw=0.6, ls=':')
    ax.set_xlabel('Position (cm)', fontsize=8.5)
    ax.set_ylabel('Mean z-scored rate', fontsize=8.5)
    ax.set_title(f'{grp_label} — profile by ML bin\n(lateral → medial: yellow → purple)',
                 fontsize=9, fontweight='bold')
    ax.legend(fontsize=6, frameon=False, loc='upper right')
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(labelsize=7)

    # ── Row 1: heatmap (ML bin × position) ───────────────────────────────────
    ax = axes[1, col]
    mat = np.full((N_ML_BINS, N_BINS), np.nan)
    ns  = []
    for bi in range(N_ML_BINS):
        lo, hi = ml_q_edges[bi], ml_q_edges[bi + 1]
        mask   = (grp['ml'] >= lo) & (grp['ml'] < hi)
        sub_X  = X_z[grp.index[mask.values], :]
        if mask.sum() >= 3:
            mat[bi] = sub_X.mean(axis=0)
        ns.append(mask.sum())

    vabs = np.nanpercentile(np.abs(mat), 95)
    im   = ax.imshow(mat, aspect='auto', origin='lower',
                     extent=[0, TRACK_LEN, 0.5, N_ML_BINS + 0.5],
                     cmap='RdBu_r', vmin=-vabs, vmax=vabs, interpolation='nearest')
    plt.colorbar(im, ax=ax, label='Mean z-score', shrink=0.85)
    ax.set_yticks(range(1, N_ML_BINS + 1))
    ax.set_yticklabels([f'{ml_labels[i]}\nn={ns[i]}' for i in range(N_ML_BINS)],
                       fontsize=5.5)
    ax.set_xlabel('Position (cm)', fontsize=8.5)
    ax.set_ylabel('ML bin  (bottom=lateral, top=medial)', fontsize=7.5)
    ax.set_title(f'{grp_label} — z-scored rate heatmap', fontsize=9, fontweight='bold')
    ax.tick_params(labelsize=7)

fig.suptitle('Mean z-scored VR firing rate profile across the ML axis',
             fontsize=11, fontweight='bold')
plt.savefig('profile_ml_bins.pdf', bbox_inches='tight', dpi=150)
plt.show()

### 2 — Which PCs drive the ML gradient, and what do they represent?

For each PC significantly correlated with ML position:
- Show its loading (the spatial pattern it captures)
- Show median PC score per ML bin for GC and NG
- Reconstruct the "medial" vs "lateral" profile difference by projecting the loading

In [ ]:
# ── Identify ML-significant PCs ───────────────────────────────────────────────
N_PC_TEST  = 10
sig_pcs    = []                       # (pc_index_0based, rho, pval)
for i in range(N_PC_TEST):
    r, p = spearmanr(df_z['ml'], df_z[f'PC{i+1}'])
    if p < 0.05:
        sig_pcs.append((i, r, p))

print(f'ML-significant PCs (p<0.05): {[f"PC{i+1} ρ={r:.2f} p={p:.3f}" for i,r,p in sig_pcs]}')

if not sig_pcs:
    print('No PCs significantly correlated with ML at p<0.05 — nothing to plot.')
else:
    N_SIG   = len(sig_pcs)
    N_BINS_ML = 8
    ml_edges   = np.linspace(df_z['ml'].quantile(0.02),
                              df_z['ml'].quantile(0.98), N_BINS_ML + 1)
    ml_centres = 0.5 * (ml_edges[:-1] + ml_edges[1:])

    fig = plt.figure(figsize=(5 * N_SIG, 10))
    outer_gs = fig.add_gridspec(3, N_SIG, hspace=0.55, wspace=0.35)

    for col, (pc_i, rho, pval) in enumerate(sig_pcs):
        pc_col  = f'PC{pc_i+1}'
        loading = pca.components_[pc_i]

        # ── Row 0: loading ────────────────────────────────────────────────────
        ax = fig.add_subplot(outer_gs[0, col])
        ax.fill_between(BIN_CENTRES, loading, alpha=0.40, color='#37474F', lw=0)
        ax.plot(BIN_CENTRES, loading, color='#1A237E', lw=1.2)
        ax.axhline(0, color='#90A4AE', lw=0.6, ls=':')
        ax.set_title(f'PC{pc_i+1} loading\n({var_exp[pc_i]*100:.1f}% var)',
                     fontsize=9, fontweight='bold')
        ax.set_xlabel('Position (cm)', fontsize=8)
        ax.set_ylabel('Loading weight', fontsize=8)
        ax.spines[['top', 'right']].set_visible(False)
        ax.tick_params(labelsize=7)

        # ── Row 1: median PC score per ML bin — GC vs NG ─────────────────────
        ax = fig.add_subplot(outer_gs[1, col])
        for ct in ['GC', 'NG']:
            sub  = df_z[df_z['cell_type'] == ct]
            meds = []
            sems = []
            for lo, hi in zip(ml_edges[:-1], ml_edges[1:]):
                grp = sub.loc[(sub['ml'] >= lo) & (sub['ml'] < hi), pc_col]
                meds.append(grp.median() if len(grp) >= 3 else np.nan)
                sems.append(grp.std() / np.sqrt(len(grp)) if len(grp) >= 3 else np.nan)
            meds = np.array(meds); sems = np.array(sems)
            ax.fill_between(ml_centres, meds - sems, meds + sems,
                            alpha=0.20, color=TYPE_COLOURS[ct], lw=0)
            ax.plot(ml_centres, meds, 'o-', color=TYPE_COLOURS[ct],
                    lw=1.6, ms=4, label=ct)
        ax.axhline(0, color='#90A4AE', lw=0.6, ls=':')
        ax.set_xlabel('ML position (µm)', fontsize=8)
        ax.set_ylabel(f'Median {pc_col} score', fontsize=8)
        ax.set_title(f'{pc_col} score vs ML\nρ={rho:.2f}  p={pval:.3f}',
                     fontsize=9, fontweight='bold')
        ax.legend(fontsize=8, frameon=False)
        ax.spines[['top', 'right']].set_visible(False)
        ax.tick_params(labelsize=7)

        # ── Row 2: medial vs lateral mean z-scored profile ────────────────────
        ax = fig.add_subplot(outer_gs[2, col])
        Q25_ML = df_z['ml'].quantile(0.25)
        Q75_ML = df_z['ml'].quantile(0.75)
        for ct in ['GC', 'NG']:
            sub   = df_z[df_z['cell_type'] == ct]
            lat_m = sub['ml'] <= Q25_ML
            med_m = sub['ml'] >= Q75_ML
            for mask, lbl, ls in [(lat_m, 'lateral', '--'), (med_m, 'medial', '-')]:
                idx   = sub.index[mask.values]
                if len(idx) < 3:
                    continue
                pr    = X_z[idx, :].mean(axis=0)
                sem   = X_z[idx, :].std(axis=0) / np.sqrt(len(idx))
                ax.fill_between(BIN_CENTRES, pr - sem, pr + sem,
                                alpha=0.15, color=TYPE_COLOURS[ct], lw=0)
                ax.plot(BIN_CENTRES, pr, color=TYPE_COLOURS[ct], lw=1.4, ls=ls,
                        label=f'{ct} {lbl} (n={len(idx)})')
        ax.axhline(0, color='#90A4AE', lw=0.6, ls=':')
        ax.set_xlabel('Position (cm)', fontsize=8)
        ax.set_ylabel('Mean z-scored rate', fontsize=8)
        ax.set_title(f'Medial vs lateral mean profile\n(bottom/top ML quartile)',
                     fontsize=9, fontweight='bold')
        ax.legend(fontsize=6.5, frameon=False)
        ax.spines[['top', 'right']].set_visible(False)
        ax.tick_params(labelsize=7)

    fig.suptitle('ML-significant PCs: loading → score gradient → profile reconstruction',
                 fontsize=11, fontweight='bold')
    plt.savefig('profile_ml_pc_decomposition.pdf', bbox_inches='tight', dpi=150)
    plt.show()